# Ma base de données : modèle à adapter

Ce notebook reprend les recettes 10 à 13 de la séquence 5, dans l'ordre. Remplace tout ce qui est marqué ✏️ par les éléments de ton projet.

Avant de commencer : MySQL démarré, et ton fichier `.env` créé, comme en séquence 5.

In [1]:
%pip install pymysql python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text, URL

load_dotenv("../.env")                      # lit ton identifiant et ton mot de passe

def connecter(nom_base=None):
    """La connexion à MySQL : au serveur seul, ou à une base précise."""
    url = URL.create("mysql+pymysql", username=os.getenv("MYSQL_UTILISATEUR"),
                     password=os.getenv("MYSQL_MOT_DE_PASSE"), host="localhost", port=3306, database=nom_base)
    return create_engine(url)

def executer_sql(moteur, chemin):
    """Exécute un fichier .sql, instruction par instruction."""
    with open(chemin, encoding="utf-8") as fichier:
        lignes = [ligne for ligne in fichier if not ligne.strip().startswith("--")]
    instructions = [i.strip() for i in "".join(lignes).split(";") if i.strip()]
    with moteur.begin() as connexion:
        for instruction in instructions:
            connexion.execute(text(instruction))
    print(len(instructions), "instructions exécutées")

serveur = connecter()                       
base = connecter("airbus_supplychain")                 
print("Connexions prêtes.")

Connexions prêtes.


## 1 · Modéliser


| Entité | Attributs | Clé primaire | Relations |
| --- | --- | --- | --- |
| PIECE | part_id, part_family, criticality_class, unit_cost, lead_time_days, supplier_id_primary, supplier_risk_class | part_id | Une pièce peut avoir plusieurs commandes, ou aucune. Une pièce a une ligne d'historique chaque semaine et sur chaque site. |
| COMMANDE | po_id, part_id (clé étrangère vers PIECE), supplier_id, site_id, order_date, promised_date, receipt_date, ordered_qty, received_qty, annee, mois | po_id | Chaque commande concerne exactement une pièce : elle contient le numéro de la pièce (part_id), qui renvoie à sa fiche. |
| HISTORIQUE_HEBDO | id, part_id (clé étrangère vers PIECE), site_id, date, consumption_qty, on_hand_qty, backorder_qty, annee, mois | id (numéro automatique). Unique : (site_id, part_id, date) | Chaque ligne concerne exactement une pièce : elle contient part_id, qui renvoie à sa fiche. |
| GSCPI_MENSUEL | date, gscpi, annee, mois | date | Aucun lien par numéro : relié aux autres tables par annee + mois, avec une jointure SQL. |
| TAUX_CHANGE_MENSUEL| annee, mois, taux_eur_usd_moyen | date | Aucun lien par numéro : relié aux autres tables par annee + mois, avec une jointure SQL. |

## 2 · Créer les tables

✏️ Adapte `creer_tables_modele.sql`, range-le dans un dossier `sql/` de ton projet, puis exécute-le.

In [3]:
executer_sql(serveur, "../sql/creer_base.sql")         
pd.read_sql("SHOW TABLES", base)

8 instructions exécutées


,Tables_in_airbus_supplychain
0,commandes
1,gscpi_mensuel
2,historique_hebdo
3,pieces
4,taux_change_mensuel


## 3 · Alimenter : les tables parentes d'abord

In [4]:
# ÉTAPE 3 : alimenter ma base. D'abord la table PARENTE, puis les tables ENFANTS.
# Une commande renvoie à une pièce par part_id : si la pièce n'est pas encore dans la base, MySQL refuse la commande.

# --- 1. La table parente : pieces ---
pieces = pd.read_csv("../data/processed/parts_master.csv")      # mon fichier préparé

# Je garde exactement les 7 colonnes de la table pieces, dans le même ordre
# (je laisse de côté shelf_life_days et is_repairable, comme décidé)
pieces = pieces[["part_id", "part_family", "criticality_class", "unit_cost", "lead_time_days", "supplier_id_primary", "supplier_risk_class"]]

# append = j'ajoute les lignes dans la table qui existe déjà, avec ses clés et ses règles (jamais replace)
pieces.to_sql("pieces", base, if_exists="append", index=False)
print("pieces :", len(pieces), "lignes envoyées")             # je dois obtenir 300

pieces : 300 lignes envoyées


In [5]:
# --- 2. Une table enfant : commandes ---
# parse_dates : je relis les 3 colonnes de dates comme de vraies dates
commandes = pd.read_csv("../data/processed/commandes_enrichies.csv", parse_dates=["order_date", "promised_date", "receipt_date"])

# Le fichier contient aussi les infos de la pièce (famille, criticité...) : elles sont déjà rangées dans pieces.
# Je ne garde que les 11 colonnes de la table commandes
commandes = commandes[["po_id", "part_id", "supplier_id", "site_id", "order_date", "promised_date", "receipt_date", "ordered_qty", "received_qty", "annee", "mois"]]

commandes.to_sql("commandes", base, if_exists="append", index=False)
print("commandes :", len(commandes), "lignes envoyées")       # je dois obtenir 29666

commandes : 29666 lignes envoyées


In [6]:
# --- 3. Une autre table enfant : historique_hebdo ---
historique = pd.read_csv("../data/processed/historique_hebdo.csv", parse_dates=["date"])

# Je garde les colonnes de la table, sans la colonne id : c'est MySQL qui l'attribue automatiquement
historique = historique[["part_id", "site_id", "date", "consumption_qty", "on_hand_qty", "backorder_qty", "annee", "mois"]]

# chunksize=10000 : le fichier est gros, je l'envoie par paquets de 10 000 lignes
historique.to_sql("historique_hebdo", base, if_exists="append", index=False, chunksize=10000)
print("historique_hebdo :", len(historique), "lignes envoyées")   # je dois obtenir 280800

historique_hebdo : 280800 lignes envoyées


In [7]:
# --- 4. Les deux indices externes : ils ne dépendent d'aucune pièce, je peux les ranger dans n'importe quel ordre ---
gscpi = pd.read_csv("../data/processed/gscpi_mensuel.csv", parse_dates=["date"])
gscpi.to_sql("gscpi_mensuel", base, if_exists="append", index=False)
print("gscpi_mensuel :", len(gscpi), "lignes envoyées")

taux = pd.read_csv("../data/processed/taux_change_mensuel.csv")
taux.to_sql("taux_change_mensuel", base, if_exists="append", index=False)
print("taux_change_mensuel :", len(taux), "lignes envoyées")         # je dois obtenir 36

gscpi_mensuel : 346 lignes envoyées
taux_change_mensuel : 36 lignes envoyées


## 4 · Contrôler après l'import

In [8]:
# Requêtes envoyées à MySQL afin de contrôler ma base
def compter(requete):
    return pd.read_sql(requete, base).iloc[0, 0]

controles = {
    # --- volume : y a-t-il autant de lignes dans la base que dans mes fichiers ? ---
    "volume : pieces": compter("SELECT COUNT(*) FROM pieces") == len(pieces),
    "volume : commandes": compter("SELECT COUNT(*) FROM commandes") == len(commandes),
    "volume : historique_hebdo": compter("SELECT COUNT(*) FROM historique_hebdo") == len(historique),
    "volume : gscpi_mensuel": compter("SELECT COUNT(*) FROM gscpi_mensuel") == len(gscpi),
    "volume : taux_change_mensuel": compter("SELECT COUNT(*) FROM taux_change_mensuel") == len(taux),

    # --- intégrité : toute commande (et toute ligne d'historique) a-t-elle sa pièce ? ---
    # LEFT JOIN ... IS NULL : je cherche les lignes qui n'ont pas de pièce correspondante, il doit y en avoir 0
    "intégrité : aucune commande sans pièce": compter("""SELECT COUNT(*) FROM commandes c
                                                         LEFT JOIN pieces p ON c.part_id = p.part_id
                                                         WHERE p.part_id IS NULL""") == 0,
    "intégrité : aucun historique sans pièce": compter("""SELECT COUNT(*) FROM historique_hebdo h
                                                          LEFT JOIN pieces p ON h.part_id = p.part_id
                                                          WHERE p.part_id IS NULL""") == 0,

    # --- cohérence : les totaux dans la base sont-ils les mêmes que dans mes fichiers ? ---
    "cohérence : même total de quantités commandées": compter("SELECT SUM(ordered_qty) FROM commandes") == commandes["ordered_qty"].sum(),
    "cohérence : même total de pièces en attente": compter("SELECT SUM(backorder_qty) FROM historique_hebdo") == historique["backorder_qty"].sum(),
    "cohérence : mêmes dates de commande (première date)": compter("SELECT MIN(order_date) FROM commandes") == commandes["order_date"].min().date(),

    # --- croisement : un seul taux par mois, sur les 36 mois de mes données ---
    "croisement : un seul taux par mois, 36 mois": compter("SELECT COUNT(*) FROM taux_change_mensuel WHERE annee BETWEEN 2022 AND 2024") == 36,
}
for nom, resultat in controles.items():
    print("OK    " if resultat else "ÉCHEC ", nom)

OK     volume : pieces
OK     volume : commandes
OK     volume : historique_hebdo
OK     volume : gscpi_mensuel
OK     volume : taux_change_mensuel
OK     intégrité : aucune commande sans pièce
OK     intégrité : aucun historique sans pièce
OK     cohérence : même total de quantités commandées
OK     cohérence : même total de pièces en attente
OK     cohérence : mêmes dates de commande (première date)
OK     croisement : un seul taux par mois, 36 mois


## 5 · Le RGPD

**Mon projet ne contient aucune donnée personnelle.** Le jeu de données Kaggle est entièrement synthétique : il ne représente aucune entreprise réelle ni aucune personne. Il ne contient ni nom, ni adresse e-mail, ni numéro de téléphone : seulement des codes (pièces, fournisseurs, sites, commandes). Mes deux autres sources, le GSCPI et les taux de change, sont des indicateurs publics agrégés : ils ne concernent aucun individu. Aucune fiche RGPD n'est donc nécessaire.

## 6 · Documenter

Le dictionnaire de données de ma base est dans `docs/dictionnaire_donnees.md`.